# QuEstX: Circuit-to-Graph Preprocessing Pipeline

This notebook converts quantum circuits in OpenQASM format into PyTorch Geometric (PyG) graphs, where nodes represent quantum gates and edges encode qubit dependencies.

1. Library Installation and Import

Initializes the environment with necessary graph neural network (GNN) and quantum computing libraries.

In [ ]:
# Install essential libraries for GNNs, quantum simulation, and hardware metrics
%pip install torch_geometric torchpack
%pip install qiskit qiskit_ibm_runtime networkx 

import pickle
import random
import torch
import networkx as nx
import string
from qiskit import QuantumCircuit, transpile
from qiskit.converters import circuit_to_dag
from qiskit.dagcircuit import DAGInNode, DAGOpNode, DAGOutNode
from qiskit.transpiler.passes import RemoveBarriers
from qiskit_ibm_runtime.fake_provider import FakeJakartaV2, FakeLimaV2, FakeManilaV2
from torch_geometric.utils.convert import from_networkx

2. Data Loading

Loads the raw PST-labeled dataset generated by the simulation scripts (e.g., D1 or D2).

In [ ]:
def load_pick(file_name):
    """Helper to load serialized pickle data."""
    file_open = open(file_name, "rb")
    data = pickle.load(file_open)
    file_open.close()
    return data

# Path to the raw PST-labeled dataset (OpenQASM strings + Calibration dict + PST)
raw_data_path = '/raw_data_qasm/dataset_D2.data'
data = load_pick(raw_data_path)

3. Circuit-to-DAG Converter (PyG Integration)

This section contains the core logic for encoding quantum circuits as directed acyclic graphs (DAGs). It extracts gate types, qubit indices, and hardware calibration parameters (T1​, T2​, gate error) into node feature tensors.

In [ ]:
# Mapping of gates to numerical IDs for one-hot encoding
GATE_DICT = {'id': 0, 'u1': 1, 'u2': 2, 'u3': 3,
    'x': 4, 'sx': 5, 'y': 6, 'z': 7, 'h': 8, 's': 9, 'sdg': 10, 't': 11, 'tdg': 12,
    'rx': 13, 'ry': 14, 'rz': 15,
    'cx': 16, 'cy': 17, 'cz': 18, 'ch': 19, 'swap': 20,
    'measure': 21, 'barrier': 22}
NUM_ERROR_DATA = 7  # Calibration features: T1, T2, gate/readout errors
NUM_NODE_TYPE = 2 + len(GATE_DICT)

def get_global_features(circ):
    """Computes circuit-level features: depth, width, and gate counts."""
    data = torch.zeros((1, 2 + len(GATE_DICT))) # Adjusted size to accommodate all gate counts
    data[0][0] = circ.depth()
    data[0][1] = circ.width()
    for key in GATE_DICT:
        if key in circ.count_ops():
            data[0][2 + GATE_DICT[key]] = circ.count_ops()[key]
    return data

def data_generator(node, mydict):
    if isinstance(node, DAGInNode):
        qubit_idx = int(node.wire._index)
        return "in", [qubit_idx], [mydict["qubit"][qubit_idx]]
    elif isinstance(node, DAGOutNode):
        qubit_idx = int(node.wire._index)
        return "out", [qubit_idx], [mydict["qubit"][qubit_idx]]
    elif isinstance(node, DAGOpNode):
        name = node.name
        qargs = node.qargs
        qubit_list = [qubit._index for qubit in qargs]
        mylist = [mydict["qubit"][qubit_idx] for qubit_idx in qubit_list]
        # The following line can cause a KeyError if name is not in mydict["gate"][tuple(qubit_list)]
        # Added .get() to handle cases where a gate might not have noise info
        gate_noise_info = mydict["gate"][tuple(qubit_list)].get(name, 0.0) # Default to 0.0 if not found
        mylist.append(gate_noise_info)
        return (name, qubit_list, mylist)
    else:
        raise NotImplementedError("Unknown node type")

def circ_to_dag_with_data(circ, mydict, max_num_qubits):
    """
    Converts a Qiskit circuit into a PyG graph.
    - Encodes gate type (one-hot)
    - Encodes qubit index (one-hot)
    - Injects hardware-aware noise features
    """
    circ = circ.copy()
    circ = RemoveBarriers()(circ)
    dag_circuit = circuit_to_dag(circ)
    dag_nodes = list(dag_circuit.nodes())
    used_qubit_idx_list = {} 
    used_qubit_idx = 0

    # NetworkX graph construction using only public methods
    G_nx = nx.DiGraph()
    for node_idx, node in enumerate(dag_nodes):
        # Extract metadata (type, qubits, calibration) via data_generator helper
        node_type, qubit_idxs, noise_info = data_generator(node, mydict)
        for qubit_idx in qubit_idxs:
            if qubit_idx not in used_qubit_idx_list:
                used_qubit_idx_list[qubit_idx] = used_qubit_idx
                used_qubit_idx += 1
        # Use max_num_qubits to define a consistent size for the data tensor
        data = torch.zeros(NUM_NODE_TYPE + max_num_qubits + NUM_ERROR_DATA + 1)
        if node_type == "in":
            data[0] = 1
            # Map qubit index to the correct slot based on max_num_qubits
            data[NUM_NODE_TYPE + used_qubit_idx_list[qubit_idxs[0]]] = 1
            data[NUM_NODE_TYPE + max_num_qubits] = noise_info[0].get("T1", 0.0) # Added .get() with default
            data[NUM_NODE_TYPE + max_num_qubits + 1] = noise_info[0].get("T2", 0.0) # Added .get() with default
        elif node_type == "out":
            data[1] = 1
            # Map qubit index to the correct slot based on max_num_qubits
            data[NUM_NODE_TYPE + used_qubit_idx_list[qubit_idxs[0]]] = 1
            data[NUM_NODE_TYPE + max_num_qubits] = noise_info[0].get("T1", 0.0) # Added .get() with default
            data[NUM_NODE_TYPE + max_num_qubits + 1] = noise_info[0].get("T2", 0.0) # Added .get() with default
            data[NUM_NODE_TYPE + max_num_qubits + 5] = noise_info[0].get("prob_meas0_prep1", 0.0) # Added .get() with default
            data[NUM_NODE_TYPE + max_num_qubits + 6] = noise_info[0].get("prob_meas1_prep0", 0.0) # Added .get() with default
        else:
            # Ensure node_type exists in GATE_DICT
            if node_type in GATE_DICT:
                data[2 + GATE_DICT[node_type]] = 1
            else:
                # Handle unexpected gate types, e.g., print a warning or skip
                print(f"Warning: Unknown gate type '{node_type}' encountered and skipped.")
                continue # Skip processing this node if it's an unknown gate type

            for i in range(len(qubit_idxs)):
                # Map qubit index to the correct slot based on max_num_qubits
                data[NUM_NODE_TYPE + used_qubit_idx_list[qubit_idxs[i]]] = 1
                # Added .get() with default values for T1 and T2 to prevent KeyError
                data[NUM_NODE_TYPE + max_num_qubits + 2 * i] = noise_info[i].get("T1", 0.0)
                data[NUM_NODE_TYPE + max_num_qubits + 2 * i + 1] = noise_info[i].get("T2", 0.0)

            # noise_info[-1] refers to the gate error, which is already defaulted to 0.0 in data_generator
            data[NUM_NODE_TYPE + max_num_qubits + 4] = noise_info[-1]
        data[-1] = node_idx
        G_nx.add_node(node_idx, x=data)

    # Add edges
    for edge in dag_circuit.edges():
        src_idx = dag_nodes.index(edge[0])
        tgt_idx = dag_nodes.index(edge[1])
        G_nx.add_edge(src_idx, tgt_idx)

    mapping = dict(zip(G_nx.nodes(), string.ascii_lowercase))
    G_nx = nx.relabel_nodes(G_nx, mapping)

    global_features = get_global_features(circ)
    liu_features = get_liu_features(dag_nodes, used_qubit_idx_list, mydict, max_num_qubits)
    return networkx_torch_convert(G_nx, global_features, liu_features)

def get_liu_features(dag_list, used_qubit_idx_list, mydict, max_num_qubits_in_dataset):
    # The first max_num_qubits_in_dataset elements are for single qubit gate counts
    # The next max_num_qubits_in_dataset * max_num_qubits_in_dataset elements are for two-qubit gate counts
    lius_feature_size = max_num_qubits_in_dataset + (max_num_qubits_in_dataset * max_num_qubits_in_dataset)
    lius_feature = torch.zeros((1, lius_feature_size))

    for node_idx, node in enumerate(dag_list):
        node_type, qubit_idxs, noise_info = data_generator(node, mydict)
        if node_type in ("rz", "x", "sx"):
            # Index for single qubit gates. These are directly mapped to the qubit index.
            lius_feature[0][used_qubit_idx_list[qubit_idxs[0]]] += 1
        elif node_type == "cx":
            # Index for two-qubit gates (CX).
            # These features start after the single-qubit features.
            q0_mapped_idx = used_qubit_idx_list[qubit_idxs[0]]
            q1_mapped_idx = used_qubit_idx_list[qubit_idxs[1]]
            # Calculate the 2D index for the CX gate based on the two qubit indices
            cx_feature_offset = max_num_qubits_in_dataset
            cx_index_in_matrix = (q0_mapped_idx * max_num_qubits_in_dataset) + q1_mapped_idx
            lius_feature[0][cx_feature_offset + cx_index_in_matrix] += 1
    return lius_feature

def networkx_torch_convert(G_nx, global_features, liu_features):
    x = torch.stack([data["x"] for _, data in G_nx.nodes(data=True)], dim=0)
    G = from_networkx(G_nx)
    G.x = x
    G.global_features = global_features
    G.liu_features = liu_features
    return G

def build_my_noise_dict(prop):
    mydict = {}
    mydict["qubit"] = {}
    mydict["gate"] = {}
    for i, qubit_prop in enumerate(prop["qubits"]):
        mydict["qubit"][i] = {}
        for item in qubit_prop:
            if item["name"] == "T1":
                mydict["qubit"][i]["T1"] = item["value"]
            elif item["name"] == "T2":
                mydict["qubit"][i]["T2"] = item["value"]
            elif item["name"] == "prob_meas0_prep1":
                mydict["qubit"][i]["prob_meas0_prep1"] = item["value"]
            elif item["name"] == "prob_meas1_prep0":
                mydict["qubit"][i]["prob_meas1_prep0"] = item["value"]
    for gate_prop in prop["gates"]:
        # Only add gates that are explicitly in GATE_DICT
        if gate_prop["gate"] not in GATE_DICT:
            continue
        qubit_list = tuple(gate_prop["qubits"])
        if qubit_list not in mydict["gate"]:
            mydict["gate"][qubit_list] = {}
        for item in gate_prop["parameters"]:
            if item["name"] == "gate_error":
                mydict["gate"][qubit_list][gate_prop["gate"]] = item["value"]
    return mydict

evalmode = False

def raw_pyg_converter(dataset, max_num_qubits): # Modified to accept max_num_qubits
    pygdataset = []
    for data in dataset:
        circ = QuantumCircuit.from_qasm_str(data[0])
        # Pass max_num_qubits to ensure consistent tensor sizes
        dag = circ_to_dag_with_data(circ, data[1], max_num_qubits)
        dag.y = data[2]
        pygdataset.append(dag)
    return pygdataset

def load_data_and_save(raw_file_path):
    with open(raw_file_path, "rb") as file:
        data = pickle.load(file)

    # Determine the maximum number of qubits across all circuits
    max_num_qubits = 0
    for item in data:
        circ = QuantumCircuit.from_qasm_str(item[0])
        max_num_qubits = max(max_num_qubits, circ.num_qubits)

    pyg_data = raw_pyg_converter(data, max_num_qubits) # Pass max_num_qubits
    random.shuffle(pyg_data)

    file_name = raw_file_path.split("/")[-1]

    import os
    os.makedirs("data/pyg_data", exist_ok=True)

    with open(f"data/pyg_data/{file_name}", "wb") as file:
        pickle.dump(pyg_data, file)
    print(f"Saved pyg_data: {len(pyg_data)} samples")
    return pyg_data

def load_normalized_data(file_name):
    with open(f"data/normalized_data/{file_name}", "rb") as file:
        data = pickle.load(file)
    print(f"Loaded normalized data: {len(data)} samples")
    return data

def normalize_data(file_name):
    """
    Performs Z-score normalization across the entire dataset.
    Saves metadata (means/stds) to ensure consistent scaling during evaluation.
    """
    with open(f"data/pyg_data/{file_name}", "rb") as file:
        data = pickle.load(file)

    if evalmode:
        with open(f"data/normalized_data/{file_name}meta", "rb") as file:
            meta = pickle.load(file)
        print("Loaded meta:", meta)
        for k, dag in enumerate(data):
            dag.x = (dag.x - meta[0]) / (1e-8 + meta[1])
            dag.global_features = (dag.global_features - meta[2]) / (1e-8 + meta[3])
            dag.liu_features = (dag.liu_features - meta[4]) / (1e-8 + meta[5])
    else:
        all_features = None
        for k, dag in enumerate(data):
            if not k:
                all_features = dag.x
                global_features = dag.global_features
                liu_features = dag.liu_features
            else:
                all_features = torch.cat([all_features, dag.x])
                global_features = torch.cat([global_features, dag.global_features])
                liu_features = torch.cat([liu_features, dag.liu_features])

        means = all_features.mean(0)
        stds = all_features.std(0)
        means_gf = global_features.mean(0)
        stds_gf = global_features.std(0)
        means_liu = liu_features.mean(0)
        stds_liu = liu_features.std(0)
        for k, dag in enumerate(data):
            dag.x = (dag.x - means) / (1e-8 + stds)
            dag.global_features = (dag.global_features - means_gf) / (1e-8 + stds_gf)
            dag.liu_features = (dag.liu_features - means_liu) / (1e-8 + stds_liu)

        import os
        os.makedirs("data/normalized_data", exist_ok=True)

        with open(f"data/normalized_data/{file_name}meta", "wb") as file:
            pickle.dump([means, stds, means_gf, stds_gf, means_liu, stds_liu], file)
    with open(f"data/normalized_data/{file_name}", "wb") as file:
        pickle.dump(data, file)
    print(f"Normalized and saved data: {len(data)} samples")


# --- MAIN EXECUTION LOOP ---

if __name__ == "__main__":
    
    raw_data_path = "raw_data_qasm/dataset_D2.data"

    pyg_data = load_data_and_save(raw_data_path)

    file_name = raw_data_path.split("/")[-1]
    normalize_data(file_name)

    norm_data = load_normalized_data(file_name)

    print(norm_data[0].global_features)


### Usage Summary for Others

    Requirement: Ensure your raw dataset is a .data file (pickle) containing a list of [qasm_string, calibration_dict, pst_label].  

    Output: The script produces a normalized .data file in data/normalized_data/. This file is the direct input for the QuEstX Model training code.

    Key Finding Support: Note the max_qubits parameter. In Experiment E3, we found that fixed-size one-hot qubit encoding causes OOD failure when moving from 10 to 18 qubits. This script handles that padding logic.